In [1]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

import time
import re
import requests
import json
import chromadb
from typing import List, Dict, Any

from configs.setting import settings
from configs.GetConfig import config
from src.a_ingestion.a1_loader import SupabaseDataLoader
from src.b_indexing.b0_vector_db import ChromaVectorDatabase
from src.LLMService import LLMService

In [2]:
SuperbaseDataLoader = SupabaseDataLoader()
products = SuperbaseDataLoader.load_products()
policies = SuperbaseDataLoader.load_policies()

[Loader] Đang tải dữ liệu sản phẩm từ Supabase...
  -> Đã tải batch 0 - 999 (1000 sản phẩm)
  -> Đã tải batch 1000 - 1324 (325 sản phẩm)
[Loader] Đã tải thành công tổng cộng 1325 sản phẩm.
[Loader] Đang tải dữ liệu chính sách từ Supabase...
[Loader] Đã tải thành công 3 chính sách từ database.


In [3]:
# Khởi tạo kết nối ChromaDB
client = ChromaVectorDatabase()

# Tách biệt làm 2 Collection chuyên biệt
product_col = client.get_or_create_collection(name="products_collection")
policy_col = client.get_or_create_collection(name="policies_collection")

print(f"Tổng vector sản phẩm: {product_col.count()}")
print(f"Tổng vector chính sách: {policy_col.count()}")

Tổng vector sản phẩm: 1325
Tổng vector chính sách: 3


In [2]:
# 1. Import trực tiếp các hàm Tool từ d_tools
from src.d_tools import product_search, policy_search

# ==========================================
# 🔍 TEST 1: Tìm kiếm sản phẩm (Product Search)
# ==========================================
print("=== 💻 CHẠY THỬ PRODUCT SEARCH ===")
product_result = product_search(key_word="laptop HP dưới 20 triệu", limit=3)
print(product_result)

print("\n" + "="*50 + "\n")

# ==========================================
# 📄 TEST 2: Tìm kiếm chính sách (Policy Search)
# ==========================================
print("=== 📋 CHẠY THỬ POLICY SEARCH ===")
policy_result = policy_search(key_word="đổi trả", limit=2)
print(policy_result)


=== 💻 CHẠY THỬ PRODUCT SEARCH ===
Product Laptop HP 15-DY2093/2193DX 405F7UA
- Brand: HP
- Price: 14,990,000 VNĐ
- Score: 0.1999
- Details:
Sản phẩm: Laptop HP 15-DY2093/2193DX 405F7UA

Thương hiệu: HP | Danh mục: laptop

Thông tin giá & Kho hàng:
- Giá thực tế: 14,990,000 VNĐ
- Giá gốc niêm yết: 18,990,000 VNĐ
- Mức giảm giá: 21.06%
- Tình trạng: Hết hàng

Thông số kỹ thuật chi tiết:
- Bộ vi xử lý (CPU/Chipset): Intel® Core™ i5-1135G7 (up to 4.2 GHz, 8 MB L3 cache, 4 nhân)
- Dung lượng RAM: 8GB
- Chuẩn/Loại RAM: 8 GB DDR4-2666 MHz RAM (2 x 4 GB)
- Dung lượng lưu trữ: 256 GB PCIe® NVMe™ M.2 SSD
- Kích thước màn hình: 15.6 inches
- Độ phân giải màn hình: 1080 x 1920 pixels (FullHD)
- Công nghệ màn hình: Màn hình chống chói, 250 nits, 45% NTSC
- Loại tấm nền màn hình: Tấm nền IPS
- Dung lượng Pin: 3-cell, 41 Wh Li-ion
- Card đồ họa (GPU/VGA): Intel Iris® Xe Graphics
- Hệ điều hành: WIN 10 - Win
- Trọng lượng: 1.69 kg
- Kích thước thiết bị: 35.85 x 24.2 x 1.79 mm
- Kết nối Bluetooth: v4.2

In [3]:
# ==========================================================
# 📦 TEST 3: Tra cứu đơn hàng cá nhân (Order Lookup)
# ==========================================================
from src.d_tools import order_lookup
from app.core.security import supabase_admin_client

# 1. Tự động lấy 1 bản ghi đơn hàng thực tế trong database để làm dữ liệu test
print("Đang quét tìm đơn hàng test trong database...")
test_orders = supabase_admin_client.table("orders").select("id, user_id").limit(1).execute()

if test_orders.data:
    test_order_id = test_orders.data[0]["id"]
    test_user_id = test_orders.data[0]["user_id"]
    
    print(f"Đã tìm thấy đơn hàng test:")
    print(f"  - User ID: {test_user_id}")
    print(f"  - Order ID: {test_order_id}\n")
    
    # ----------------------------------------------------
    # CASE 2: Tra cứu danh sách đơn hàng gần đây (Không truyền order_id)
    # ----------------------------------------------------
    print("=== 📋 TEST CASE 2: DANH SÁCH ĐƠN HÀNG GẦN ĐÂY ===")
    history_result = order_lookup(current_user_id=test_user_id)
    print(history_result)
    
    print("\n" + "="*50 + "\n")
    
    # ----------------------------------------------------
    # CASE 1: Tra cứu chi tiết một đơn hàng cụ thể (Có truyền order_id)
    # ----------------------------------------------------
    print("=== 🔍 TEST CASE 1: CHI TIẾT ĐƠN HÀNG CỤ THỂ ===")
    detail_result = order_lookup(current_user_id=test_user_id, order_id=test_order_id)
    print(detail_result)

else:
    print("Thông báo: Cơ sở dữ liệu hiện tại chưa có đơn hàng nào để chạy thử test.")


Đang quét tìm đơn hàng test trong database...
Đã tìm thấy đơn hàng test:
  - User ID: ff641f26-3ada-47d0-9bf2-1f4b71654064
  - Order ID: 80a172ba-e9e0-47e5-a802-efba24dd14dd

=== 📋 TEST CASE 2: DANH SÁCH ĐƠN HÀNG GẦN ĐÂY ===
=== YOUR RECENT ORDERS ===

1. Order ID: 80a172ba-e9e0-47e5-a802-efba24dd14dd
   - Date: 2026-07-21T10:28:02.093553+00:00
   - Status: PENDING
   - Total: 15,990,000 VNĐ
   - Products: OPPO Reno16 F 5G 8GB 256GB


=== 🔍 TEST CASE 1: CHI TIẾT ĐƠN HÀNG CỤ THỂ ===
=== ORDER DETAILS ===
- Order ID: 80a172ba-e9e0-47e5-a802-efba24dd14dd
- Order Date: 2026-07-21T10:28:02.093553+00:00
- Status: PENDING
- Shipping Address: B22DCKH065_Vũ Gia Khải - SĐT: 0987998685 - ĐC: ;lkjhgfd, Phường Phúc Xá, Quận Ba Đình, Thành phố Hà Nội (Ghi chú: kjhgfd)
- Total Amount: 15,990,000 VNĐ
- Purchased Items:
  - OPPO Reno16 F 5G 8GB 256GB (Quantity: 1 | Price: 15,990,000 VNĐ)
